# Cylinder + Beam FSI — Mixed-Form PINN (TF2)

Steady 2D laminar flow past a **cylinder with an attached rigid beam**.
Based on the Turek & Hron (2006) CFD benchmark geometry.

> Turek, S. & Hron, J. (2006). *Proposal for Numerical Benchmarking of
> Fluid-Structure Interaction between an Elastic Object and Laminar
> Incompressible Flow.* Lecture Notes in Computational Science and Engineering.

**Key differences from cylinder-only PINN:**
- `DelObsPT` removes points inside cylinder **and** beam rectangle
- Three beam no-slip edges: top, bottom, right (left edge is attachment)
- `bc_weight = 10` (vs 2 for cylinder) — stronger BC enforcement
- Component-wise loss tracking: `loss_total`, `loss_pde`, `loss_wall`, etc.

## 1. Imports

In [ ]:
import numpy as np
import time
import pickle
import scipy.io
import scipy.optimize
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(1234)
np.random.seed(1234)

## 2. Sampling & Helpers

In [ ]:
def lhs(n_dim, n_samples, seed=None):
    """Latin Hypercube Sampling in [0,1]^n_dim."""
    rng = np.random.default_rng(seed)
    cut = np.linspace(0.0, 1.0, n_samples + 1)
    u = rng.random((n_samples, n_dim))
    a, b = cut[:n_samples], cut[1:n_samples + 1]
    rdpoints = a[:, None] + (b - a)[:, None] * u
    H = np.empty_like(rdpoints)
    for j in range(n_dim):
        H[:, j] = rdpoints[rng.permutation(n_samples), j]
    return H


def DelObsPT(XY, xc, yc, r, xb0, xb1, yb0, yb1):
    """Remove points inside cylinder OR inside beam rectangle."""
    dst    = np.sqrt((XY[:,0]-xc)**2 + (XY[:,1]-yc)**2)
    in_cyl = dst <= r
    in_beam = ((XY[:,0] >= xb0) & (XY[:,0] <= xb1) &
               (XY[:,1] >= yb0) & (XY[:,1] <= yb1))
    return XY[~(in_cyl | in_beam), :]


def preprocess_mat(dir_path):
    """Load reference solution (ANSYS Fluent .mat)."""
    data = scipy.io.loadmat(dir_path)
    X, Y = data['x'], data['y']
    vx, vy, P = data['vx'], data['vy'], data['p']
    return (X.flatten()[:,None], Y.flatten()[:,None],
            vx.flatten()[:,None], vy.flatten()[:,None],
            P.flatten()[:,None])

## 3. Model — MLP + Beam PINN

**Physics bug fix** applied here:  
`self.loss_total = []` and siblings are initialised in `__init__`,  
**not** inside the Adam print block (where they would reset every 10 iters).

In [ ]:
# Helpers
    
def DelObsPT(XY, xc, yc, r, xb0, xb1, yb0, yb1):
    """Delete points inside (cylinder) OR inside (beam rectangle)."""
    # cylinder mask
    dst = np.sqrt((XY[:,0]-xc)**2 + (XY[:,1]-yc)**2)
    in_cyl = (dst <= r)

    # beam rectangle mask
    in_beam = (XY[:,0] >= xb0) & (XY[:,0] <= xb1) & (XY[:,1] >= yb0) & (XY[:,1] <= yb1)

    return XY[~(in_cyl | in_beam), :]

def preprocess_mat(dir_path):
    """Load reference solution from Fenics or Fluent mat file.
       Expects keys: x,y,p,vx,vy"""
    data = scipy.io.loadmat(dir_path)
    X = data['x']; Y = data['y']
    P = data['p']
    vx = data['vx']; vy = data['vy']
    return (X.flatten()[:,None], Y.flatten()[:,None],
            vx.flatten()[:,None], vy.flatten()[:,None],
            P.flatten()[:,None])

def postProcess(xmin, xmax, ymin, ymax, field_ref, field_pinn, s=2, alpha=0.5, marker='o'):
    """Same plotting style as your original code."""
    x_ref, y_ref, u_ref, v_ref, p_ref = field_ref
    x_p, y_p, u_p, v_p, p_p = field_pinn

    fig, ax = plt.subplots(nrows=3, ncols=2, figsize=(7, 4))
    fig.subplots_adjust(hspace=0.2, wspace=0.2)

    def _scatter(axh, x, y, c, title, vmin=None, vmax=None):
        cf = axh.scatter(x, y, c=c, alpha=alpha, edgecolors='none',
                         cmap='rainbow', marker=marker, s=int(s),
                         vmin=vmin, vmax=vmax)
        axh.axis('square')
        for key, spine in axh.spines.items():
            spine.set_visible(False)
        axh.set_xticks([]); axh.set_yticks([])
        axh.set_xlim([xmin, xmax]); axh.set_ylim([ymin, ymax])
        axh.set_title(title)
        fig.colorbar(cf, ax=axh, fraction=0.046, pad=0.04)

    _scatter(ax[0,0], x_p, y_p, u_p, r'$u$ (PINN)')
    _scatter(ax[1,0], x_p, y_p, v_p, r'$v$ (PINN)')
    _scatter(ax[2,0], x_p, y_p, p_p, r'$p$ (PINN)', vmin=-0.25, vmax=4.0)

    _scatter(ax[0,1], x_ref, y_ref, u_ref, r'$u$ (REF)')
    _scatter(ax[1,1], x_ref, y_ref, v_ref, r'$v$ (REF)')
    _scatter(ax[2,1], x_ref, y_ref, p_ref, r'$p$ (REF)', vmin=-0.25, vmax=4.0)

    plt.show()

# -------------------------
# Keras network
# -------------------------
class MLP(tf.keras.Model):
    def __init__(self, layers_sizes, activation="tanh"):
        super().__init__()
        self.hidden = []
        for w in layers_sizes[:-1]:
            self.hidden.append(tf.keras.layers.Dense(
                w, activation=activation,
                kernel_initializer="glorot_normal"
            ))
        self.out = tf.keras.layers.Dense(
            layers_sizes[-1], activation=None,
            kernel_initializer="glorot_normal"
        )

    def call(self, x, training=False):
        z = x
        for lyr in self.hidden:
            z = lyr(z)
        return self.out(z)

# -------------------------
# Mixed-form PINN in TF2
# -------------------------
class PINN_laminar_flow_TF2:
    """
    TF2 rewrite of your TF1 class:
    Outputs: psi, p, s11, s22, s12
    u = dpsi/dy
    v = -dpsi/dx
    """
    def __init__(self, Collo, INLET, OUTLET, WALL, uv_layers, lb, ub,
                 rho=1000, mu=1, bc_weight=2.0):

        self.lb = lb.astype(np.float32).reshape(1,2)
        self.ub = ub.astype(np.float32).reshape(1,2)

        self.rho = tf.constant(rho, dtype=tf.float32)
        self.mu  = tf.constant(mu, dtype=tf.float32)
        self.bc_weight = tf.constant(bc_weight, dtype=tf.float32)

        # Data
        self.Collo = Collo.astype(np.float32)
        self.INLET = INLET.astype(np.float32)   # [x,y,u,v]
        self.OUTLET = OUTLET.astype(np.float32) # [x,y]
        self.WALL = WALL.astype(np.float32)     # [x,y]

        # Model
        hidden_widths = uv_layers[1:-1]
        out_dim = uv_layers[-1]  # 5
        self.net = MLP(hidden_widths + [out_dim], activation="tanh")

        self.loss_rec    = []
        # Loss component histories (never reset after init)
        self.loss_total  = []
        self.loss_pde    = []
        self.loss_wall   = []
        self.loss_inlet  = []
        self.loss_outlet = []

    # --- forward outputs (psi,p,s11,s22,s12) ---
    def net_psips(self, x, y):
        xy = tf.concat([x, y], axis=1)
        out = self.net(xy)
        psi = out[:,0:1]
        p   = out[:,1:2]
        s11 = out[:,2:3]
        s22 = out[:,3:4]
        s12 = out[:,4:5]
        return psi, p, s11, s22, s12

    # --- compute u,v and first derivatives with a tape ---
    def uv_and_grads(self, x, y):
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, s11, s22, s12 = self.net_psips(x, y)
            u = tape.gradient(psi, y)
            v = -tape.gradient(psi, x)

        u_x = tape.gradient(u, x)
        u_y = tape.gradient(u, y)
        v_x = tape.gradient(v, x)
        v_y = tape.gradient(v, y)

        s11_x = tape.gradient(s11, x)
        s12_x = tape.gradient(s12, x)
        s12_y = tape.gradient(s12, y)
        s22_y = tape.gradient(s22, y)

        del tape
        return u, v, p, s11, s22, s12, u_x, u_y, v_x, v_y, s11_x, s12_x, s12_y, s22_y

    # --- PDE residuals (same as your net_f) ---
    def residuals(self, x, y):
        rho = self.rho
        mu  = self.mu

        (u, v, p, s11, s22, s12,
         u_x, u_y, v_x, v_y,
         s11_x, s12_x, s12_y, s22_y) = self.uv_and_grads(x, y)

        f_u = (u*u_x + v*u_y) - s11_x/rho - s12_y/rho
        f_v = (u*v_x + v*v_y) - s12_x/rho - s22_y/rho

        f_s11 = -p + 2.0*mu*u_x - s11
        f_s22 = -p + 2.0*mu*v_y - s22
        f_s12 = mu*(u_y + v_x) - s12

        f_p = p + 0.5*(s11 + s22)
        return f_u, f_v, f_s11, f_s22, f_s12, f_p

    # --- loss ---
    def loss_fn(self, Xc, WALL, INLET, OUTLET):
        # Collocation residuals
        xc = Xc[:,0:1]; yc = Xc[:,1:2]
        fu, fv, fs11, fs22, fs12, fp = self.residuals(xc, yc)
        loss_f = (tf.reduce_mean(tf.square(fu)) +
                  tf.reduce_mean(tf.square(fv)) +
                  tf.reduce_mean(tf.square(fs11)) +
                  tf.reduce_mean(tf.square(fs22)) +
                  tf.reduce_mean(tf.square(fs12)) +
                  tf.reduce_mean(tf.square(fp)))

        # WALL: u=v=0
        xw = WALL[:,0:1]; yw = WALL[:,1:2]
        u_w, v_w, *_ = self.uv_and_grads(xw, yw)
        loss_wall = tf.reduce_mean(tf.square(u_w)) + tf.reduce_mean(tf.square(v_w))

        # INLET: match prescribed u,v
        xi = INLET[:,0:1]; yi = INLET[:,1:2]
        ui = INLET[:,2:3]; vi = INLET[:,3:4]
        u_i, v_i, *_ = self.uv_and_grads(xi, yi)
        loss_in = tf.reduce_mean(tf.square(u_i - ui)) + tf.reduce_mean(tf.square(v_i - vi))

        # OUTLET: gauge pressure condition
        xo = OUTLET[:,0:1]; yo = OUTLET[:,1:2]
        _, p_o, _, _, _ = self.net_psips(xo, yo)

        # More stable than forcing p=0 pointwise:
        loss_out = tf.square(tf.reduce_mean(p_o))

        total = loss_f + self.bc_weight*(loss_wall + loss_in + loss_out)
        return total, loss_f, loss_wall, loss_in, loss_out

    # --- Adam training loop ---
    def train_adam(self, iters=10000, lr=5e-4, print_every=10):
        opt = tf.keras.optimizers.Adam(learning_rate=lr)

        Xc = tf.convert_to_tensor(self.Collo, dtype=tf.float32)
        WALL = tf.convert_to_tensor(self.WALL, dtype=tf.float32)
        INLET = tf.convert_to_tensor(self.INLET, dtype=tf.float32)
        OUTLET = tf.convert_to_tensor(self.OUTLET, dtype=tf.float32)

        for it in range(iters):
            with tf.GradientTape() as tape:
                loss, lf, lw, lin, lout = self.loss_fn(Xc, WALL, INLET, OUTLET)
            grads = tape.gradient(loss, self.net.trainable_variables)
            opt.apply_gradients(zip(grads, self.net.trainable_variables))

            self.loss_total.append(float(loss.numpy()))
            self.loss_pde.append(float(lf.numpy()))
            self.loss_wall.append(float(lw.numpy()))
            self.loss_inlet.append(float(lin.numpy()))
            self.loss_outlet.append(float(lout.numpy()))
            if it % print_every == 0:
                print(f"It {it:6d} | loss={loss.numpy():.3e} "
                      f"| f={lf.numpy():.3e} wall={lw.numpy():.3e} "
                      f"in={lin.numpy():.3e} out={lout.numpy():.3e}")

    # --- L-BFGS with SciPy ---
    def train_lbfgs(self, maxiter=50000, maxcor=50):
        Xc = tf.convert_to_tensor(self.Collo, dtype=tf.float32)
        WALL = tf.convert_to_tensor(self.WALL, dtype=tf.float32)
        INLET = tf.convert_to_tensor(self.INLET, dtype=tf.float32)
        OUTLET = tf.convert_to_tensor(self.OUTLET, dtype=tf.float32)

        # pack/unpack weights
        def pack():
            return np.concatenate([v.numpy().reshape(-1) for v in self.net.trainable_variables]).astype(np.float64)

        def unpack(flat):
            idx = 0
            for v in self.net.trainable_variables:
                shape = v.shape
                size = int(np.prod(shape))
                v.assign(flat[idx:idx+size].reshape(shape).astype(np.float32))
                idx += size

        def loss_and_grad(flat):
            unpack(flat)
            with tf.GradientTape() as tape:
                loss, *_ = self.loss_fn(Xc, WALL, INLET, OUTLET)
            grads = tape.gradient(loss, self.net.trainable_variables)
            grad_flat = np.concatenate([g.numpy().reshape(-1) for g in grads]).astype(np.float64)
            return float(loss.numpy()), grad_flat

        x0 = pack()

        def callback(xk):
            l, grad = loss_and_grad(xk)
            total, lf, lw, lin, lout = self.loss_fn(Xc, WALL, INLET, OUTLET)
            self.loss_total.append(float(total.numpy()))
            self.loss_pde.append(float(lf.numpy()))
            self.loss_wall.append(float(lw.numpy()))
            self.loss_inlet.append(float(lin.numpy()))
            self.loss_outlet.append(float(lout.numpy()))
            print(f"L-BFGS | total={float(total):.3e} pde={float(lf):.3e}")
    
        res = scipy.optimize.minimize(
            fun=lambda w: loss_and_grad(w),
            x0=x0,
            jac=True,
            method="L-BFGS-B",
            callback=callback,
            options={"maxiter": maxiter, "maxcor": maxcor, "ftol": np.finfo(float).eps}
        )
        unpack(res.x)
        return res

    # --- Save/Load (TF2 style) ---
    def save(self, path="uvNN_tf2"):
        self.net.save_weights(path)

    def load(self, path="uvNN_tf2"):
        # build once
        dummy = tf.zeros((1,2), dtype=tf.float32)
        _ = self.net(dummy)
        self.net.load_weights(path)

    # --- predict u,v,p (and optionally stress) ---

    def predict(self, x_star, y_star):
        x = tf.convert_to_tensor(x_star.astype(np.float32))
        y = tf.convert_to_tensor(y_star.astype(np.float32))
    
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, _, _, _ = self.net_psips(x, y)
    
        u = tape.gradient(psi, y)
        v = -tape.gradient(psi, x)
        del tape
    
        return u.numpy(), v.numpy(), p.numpy()

    def predict_all(self, x_star, y_star):
        x = tf.convert_to_tensor(x_star.astype(np.float32))
        y = tf.convert_to_tensor(y_star.astype(np.float32))
    
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y])
            psi, p, s11, s22, s12 = self.net_psips(x, y)
    
        u = tape.gradient(psi, y)
        v = -tape.gradient(psi, x)
        del tape
    
        return u.numpy(), v.numpy(), p.numpy(), s11.numpy(), s22.numpy(), s12.numpy()


## 4. Problem Setup — Turek-Hron Geometry

| Parameter | Value |
|-----------|-------|
| Channel   | 1.1 × 0.41 |
| Cylinder  | center (0.2, 0.2), r = 0.05 |
| Beam      | l = 0.35, h = 0.02 |
| Beam box  | x ∈ [0.25, 0.60], y ∈ [0.19, 0.21] |
| ρ, μ      | 1.0, 0.02 (CFD1: Re ≈ 20) |
| Ū         | 0.2 m/s → U_max = 0.3 m/s |

In [ ]:
# ---- Geometry (Turek & Hron 2006) ----
L, H = 1.1, 0.41
lb   = np.array([0.0, 0.0])
ub   = np.array([L, H])
xc, yc, r = 0.2, 0.2, 0.05

# Beam bounding box
l_beam, h_beam = 0.35, 0.02
x_right, y_bot = 0.6, 0.19
x_left  = x_right - l_beam    # 0.25
y_top   = y_bot   + h_beam    # 0.21

# ---- Fluid properties (CFD1: Re ≈ 20) ----
rho  = 1.0
nu   = 0.02        # kinematic viscosity
mu   = rho * nu    # dynamic viscosity
Ubar = 0.2         # mean inlet velocity
U_max = 1.5 * Ubar # peak of parabolic profile

# ---- Boundary points ----
Nw, Ni, No, Nc, Nb = 441, 201, 201, 400, 300

wall_up = np.hstack([L*lhs(1, Nw), H*np.ones((Nw,1))])
wall_lw = np.hstack([L*lhs(1, Nw), np.zeros((Nw,1))])

y_in  = H * lhs(1, Ni)
u_in  = 4*U_max*y_in*(H-y_in)/H**2
INLET = np.hstack([np.zeros((Ni,1)), y_in, u_in, np.zeros_like(u_in)])

OUTLET = np.hstack([L*np.ones((No,1)), H*lhs(1, No)])

theta_c = 2*np.pi*lhs(1, Nc)
CYLD    = np.hstack([xc + r*np.cos(theta_c), yc + r*np.sin(theta_c)])

beam_top_pts   = np.hstack([x_left + (x_right-x_left)*lhs(1, Nb),
                             y_top*np.ones((Nb,1))])
beam_bot_pts   = np.hstack([x_left + (x_right-x_left)*lhs(1, Nb),
                             y_bot*np.ones((Nb,1))])
beam_right_pts = np.hstack([x_right*np.ones((Nb,1)),
                             y_bot + (y_top-y_bot)*lhs(1, Nb)])
BEAM = np.vstack([beam_top_pts, beam_bot_pts, beam_right_pts])
WALL = np.vstack([CYLD, BEAM, wall_up, wall_lw])

# ---- Collocation points ----
XY_c       = lb + (ub - lb) * lhs(2, 4000)
XY_refine  = np.array([0.1, 0.1]) + np.array([0.6, 0.2]) * lhs(2, 1000)
XY_c       = np.vstack([XY_c, XY_refine])
XY_c       = DelObsPT(XY_c, xc=xc, yc=yc, r=r,
                       xb0=x_left, xb1=x_right, yb0=y_bot, yb1=y_top)
XY_c       = np.vstack([XY_c, WALL, OUTLET, INLET[:,0:2]])
print('Collocation points:', XY_c.shape[0])

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.set_aspect('equal')
ax.scatter(XY_c[:,0], XY_c[:,1], s=0.5, alpha=0.08, label='Interior')
ax.scatter(WALL[:,0], WALL[:,1], s=5, alpha=0.6, label='Wall')
ax.scatter(OUTLET[:,0], OUTLET[:,1], s=8, alpha=0.6, label='Outlet')
ax.scatter(INLET[:,0], INLET[:,1], s=8, alpha=0.6, label='Inlet')
ax.legend(fontsize=7, loc='upper right')
ax.set_title('Collocation & boundary points (Cylinder + Beam)')
plt.tight_layout(); plt.show()

## 5. Training

- **Adam** 10 000 iterations at lr = 5 × 10⁻⁴
- **L-BFGS-B** up to 50 000 iterations
- `bc_weight = 10` — 5× heavier than the cylinder-only case

In [ ]:
uv_layers = [2] + 8*[40] + [5]

model = PINN_laminar_flow_TF2(
    XY_c, INLET, OUTLET, WALL, uv_layers, lb, ub,
    rho=rho, mu=mu, bc_weight=10.0
)

t0 = time.time()
model.train_adam(iters=10000, lr=5e-4, print_every=200)
n_adam = len(model.loss_total)
res    = model.train_lbfgs(maxiter=50000)
print(f'Total training time: {time.time()-t0:.1f} s')

model.save('uvNN_tf2_weights_beam1_weight10.weights.h5')

## 6. Loss Convergence

In [ ]:
iters = np.arange(len(model.loss_total))
fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(iters, model.loss_total,  'k-',  lw=1.5, label='Total')
ax.semilogy(iters, model.loss_pde,    'b--', lw=1.2, label='PDE')
ax.semilogy(iters, model.loss_wall,   'g:',  lw=1.2, label='Wall BC')
ax.semilogy(iters, model.loss_inlet,  'm:',  lw=1.2, label='Inlet BC')
ax.semilogy(iters, model.loss_outlet, 'c:',  lw=1.2, label='Outlet BC')
ax.axvline(n_adam, color='r', ls='--', lw=1.5, label='Adam → L-BFGS')
ax.set_xlabel('Iteration'); ax.set_ylabel('Loss')
ax.set_title('Loss Convergence — Cylinder + Beam (bc_weight=10)')
ax.legend(fontsize=9); ax.grid(True, which='both', ls='--', alpha=0.4)
plt.tight_layout(); plt.show()

## 7. Field Visualisation (u, v, p)

In [ ]:
# ---- Evaluation grid ----
nx, ny = 251, 101
xg = np.linspace(0.0, L, nx); yg = np.linspace(0.0, H, ny)
Xg, Yg = np.meshgrid(xg, yg)
xs = Xg.ravel()[:, None]; ys = Yg.ravel()[:, None]

dst_cyl = np.sqrt((xs - xc)**2 + (ys - yc)**2)
in_beam = ((xs[:,0] >= x_left) & (xs[:,0] <= x_right) &
            (ys[:,0] >= y_bot)  & (ys[:,0] <= y_top))
mask    = (dst_cyl[:,0] >= r) & ~in_beam
x_ev, y_ev = xs[mask], ys[mask]

u_p, v_p, p_p = model.predict(x_ev, y_ev)

# ---- Solid outlines ----
theta_plot = np.linspace(0, 2*np.pi, 400)
cyl_x = xc + r*np.cos(theta_plot); cyl_y = yc + r*np.sin(theta_plot)
beam_px = [x_left, x_right, x_right, x_left, x_left]
beam_py = [y_bot,  y_bot,  y_top,   y_top,  y_bot]

fig, axes = plt.subplots(3, 1, figsize=(11, 9), constrained_layout=True)
labels = [r'$u^*$ (PINN)', r'$v^*$ (PINN)', r'$p^*$ (PINN)']
fields = [u_p.ravel(), v_p.ravel(), p_p.ravel()]

for axh, field, label in zip(axes, fields, labels):
    sc = axh.scatter(x_ev[:,0], y_ev[:,0], c=field, s=2, cmap='jet')
    axh.plot(cyl_x, cyl_y, 'k-', lw=1.2)
    axh.plot(beam_px, beam_py, 'k-', lw=1.2)
    axh.set_aspect('equal'); axh.axis('off')
    axh.set_title(label, fontsize=11)
    plt.colorbar(sc, ax=axh, fraction=0.02, pad=0.01)

plt.show()

## 8. Pressure Distribution on Cylinder Surface

In [ ]:
from scipy.interpolate import griddata
from matplotlib.patches import Circle, FancyArrowPatch

theta_cyl = np.linspace(0, 2*np.pi, 361)
xc_pts = xc + r*np.cos(theta_cyl)
yc_pts = yc + r*np.sin(theta_cyl)
_, _, p_cyl = model.predict(xc_pts[:,None], yc_pts[:,None])

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
ax.plot(theta_cyl, p_cyl.ravel(), lw=2.5, label='PINN')
ax.set_xlim(0, 2*np.pi)
ax.set_xlabel(r'$\theta$ (rad)', fontsize=14)
ax.set_ylabel('Pressure', fontsize=14)
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels([r'$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'])
ax.legend(fontsize=12, frameon=False); ax.grid(alpha=0.4)
ax.set_title('Pressure on cylinder surface')
plt.tight_layout(); plt.show()

## 9. Drag & Lift on Cylinder + Beam

In [ ]:
import numpy as np

def _traction_from_ps(model, x, y, nx, ny):
    """
    Traction t = σ n where σ components are directly predicted:
      σ = [[s11, s12],
           [s12, s22]]
    """
    x = np.asarray(x).reshape(-1, 1).astype(np.float32)
    y = np.asarray(y).reshape(-1, 1).astype(np.float32)
    nx = np.asarray(nx).reshape(-1)
    ny = np.asarray(ny).reshape(-1)

    # predict (u, v, p, s11, s22, s12)
    _, _, _, s11, s22, s12 = model.predict_all(x, y)

    s11 = np.asarray(s11).reshape(-1)
    s22 = np.asarray(s22).reshape(-1)
    s12 = np.asarray(s12).reshape(-1)

    tx = s11*nx + s12*ny
    ty = s12*nx + s22*ny
    return tx, ty


def compute_drag_lift_cylinder_beam(
    model,
    # cylinder
    xc=0.2, yc=0.2, r=0.05,
    # beam (Turek–Hron default: length=0.35, thickness=0.02)
    beam_L=0.35, beam_h=0.02,
    # quadrature resolution
    n_theta=2000,
    n_edge=800,
    # nondimensional coefficients (optional)
    nondim=True,
    rho=1000.0,
    Ubar=0.2,
    d=None,
):
    """
    Drag/Lift on combined obstacle: cylinder + attached rigid beam.

    Beam is assumed attached at cylinder rightmost point:
        x from x_attach = xc + r  to x_end = x_attach + beam_L
        y in [yc - beam_h/2, yc + beam_h/2]

    Boundary integrated:
      - cylinder arc excluding attachment window (covered by beam)
      - beam top edge, bottom edge, right edge
      - (beam left edge NOT included; it is inside the solid attachment)

    Returns:
      if nondim: (D, L, Cd, Cl)
      else:      (D, L)
    """
    if d is None:
        d = 2.0 * r

    # ---------- 1) Cylinder arc (exclude attachment zone) ----------
    # attachment window on cylinder: y in [yc - beam_h/2, yc + beam_h/2] at x≈xc+r
    # => theta0 = arcsin((beam_h/2)/r)
    theta0 = np.arcsin(np.clip((beam_h * 0.5) / r, 0.0, 1.0))

    # sample theta in (theta0, 2π-theta0)
    th = np.linspace(theta0, 2*np.pi - theta0, n_theta, endpoint=False)
    x_c = xc + r*np.cos(th)
    y_c = yc + r*np.sin(th)
    nx_c = np.cos(th)
    ny_c = np.sin(th)
    dtheta = (2*np.pi - 2*theta0) / n_theta
    dS_c = r * dtheta

    tx_c, ty_c = _traction_from_ps(model, x_c, y_c, nx_c, ny_c)
    D_c = np.sum(tx_c) * dS_c
    L_c = np.sum(ty_c) * dS_c

    # ---------- 2) Beam exposed edges ----------
    x_attach = xc + r
    x_end    = x_attach + beam_L
    y_top    = yc + 0.5*beam_h
    y_bot    = yc - 0.5*beam_h

    # Top edge: (x_attach -> x_end), y=y_top, n=(0, +1)
    xs_top = np.linspace(x_attach, x_end, n_edge, endpoint=False)
    ys_top = np.full_like(xs_top, y_top)
    nx_top = np.zeros_like(xs_top)
    ny_top = np.ones_like(xs_top)
    ds_top = (x_end - x_attach) / n_edge

    tx_top, ty_top = _traction_from_ps(model, xs_top, ys_top, nx_top, ny_top)
    D_top = np.sum(tx_top) * ds_top
    L_top = np.sum(ty_top) * ds_top

    # Bottom edge: n=(0, -1)
    xs_bot = np.linspace(x_attach, x_end, n_edge, endpoint=False)
    ys_bot = np.full_like(xs_bot, y_bot)
    nx_bot = np.zeros_like(xs_bot)
    ny_bot = -np.ones_like(xs_bot)
    ds_bot = (x_end - x_attach) / n_edge

    tx_bot, ty_bot = _traction_from_ps(model, xs_bot, ys_bot, nx_bot, ny_bot)
    D_bot = np.sum(tx_bot) * ds_bot
    L_bot = np.sum(ty_bot) * ds_bot

    # Right edge: x=x_end, y_bot -> y_top, n=(+1, 0)
    ys_right = np.linspace(y_bot, y_top, n_edge, endpoint=False)
    xs_right = np.full_like(ys_right, x_end)
    nx_right = np.ones_like(ys_right)
    ny_right = np.zeros_like(ys_right)
    ds_right = (y_top - y_bot) / n_edge

    tx_r, ty_r = _traction_from_ps(model, xs_right, ys_right, nx_right, ny_right)
    D_r = np.sum(tx_r) * ds_right
    L_r = np.sum(ty_r) * ds_right

    # Total
    D = D_c + D_top + D_bot + D_r
    L = L_c + L_top + L_bot + L_r

    if not nondim:
        return D, L

    q = 0.5 * rho * (Ubar**2) * d
    Cd = D / q
    Cl = L / q
    return D, L, Cd, Cl


# -----------------------
# Example (CFD1)
# -----------------------
rho = 1.0
Ubar = 0.2
D, L, Cd, Cl = compute_drag_lift_cylinder_beam(
    model,
    xc=0.2, yc=0.2, r=0.05,
    beam_L=0.35, beam_h=0.02,   # Turek–Hron beam
    n_theta=3000,
    n_edge=1200,
    nondim=True,
    rho=rho,
    Ubar=Ubar,
    d=0.1
)
print(f"D={D:.6e}, L={L:.6e}  (2D force per unit depth)")
print(f"Cd={Cd:.6f}, Cl={Cl:.6f}")
print(f"Cd_{True}={0.2}, Cl_{True}={4e-3}")


D, L_f, Cd, Cl = compute_drag_lift_cylinder_beam(
    model, xc=xc, yc=yc, r=r,
    beam_L=l_beam, beam_h=h_beam,
    rho=rho, Ubar=Ubar
)
print(f'Drag D = {D:.6f}   Lift L = {L_f:.6f}')
print(f'Cd = {Cd:.4f}   Cl = {Cl:.4f}')

## 10. Save Results

In [ ]:
results = {
    'loss_total':   model.loss_total,
    'loss_pde':     model.loss_pde,
    'loss_wall':    model.loss_wall,
    'loss_inlet':   model.loss_inlet,
    'loss_outlet':  model.loss_outlet,
    'u_pred': u_p, 'v_pred': v_p, 'p_pred': p_p,
    'x_eval': x_ev, 'y_eval': y_ev,
    'drag': D, 'lift': L_f, 'Cd': Cd, 'Cl': Cl,
    'meta': {
        'rho': rho, 'mu': mu, 'Ubar': Ubar,
        'bc_weight': 10.0,
        'layers': uv_layers,
        'description': 'Steady cylinder+beam PINN (Turek-Hron CFD1)'
    }
}
with open('pinn_cylinder_beam1_results_weight10.pkl', 'wb') as f:
    pickle.dump(results, f)
print('Saved pinn_cylinder_beam1_results_weight10.pkl')